# A Reverse-Hoops-Playing Robot Arm

## Preparation

Create a virtual environment, e.g., with `uv`

```sh
sudo apt install uv
```

From the software directory, run

```sh
uv sync
```

Use the python kernel in the virtual environment for the Jupyter Notebook.

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import time

import pinocchio as pin
from pinocchio.visualize import MeshcatVisualizer
from meshcat.geometry import MeshPhongMaterial, Sphere
from meshcat.transformations import translation_matrix
import matplotlib.pyplot as plt

import casadi as ca

from utils.ball_simulation import BallSimulation


## Visualize the environment

In [ ]:


COLORS = [0xe41a1c,
          0x377eb8,
          0x4daf4a,
          0x984ea3,
          0xff7f00,
          0xffff33,
          0xa65628,
          0xf781bf,
          0x999999]



# specify a path the the urdf files and meshes
urdf_model_path = "robot_description/ur10_robot.urdf"
mesh_dir = "robot_description/meshes"

# load the robot using pinocchio
robot = pin.RobotWrapper.BuildFromURDF(urdf_model_path, mesh_dir)

LOWER_POSITION_LIMIT = robot.model.lowerPositionLimit
UPPER_POSITION_LIMIT = robot.model.upperPositionLimit
VELOCITY_LIMIT = robot.model.velocityLimit
ACCELERATION_LIMIT = 2 * np.ones(robot.model.nv)

# vizualize the robot using meshcat
viz = MeshcatVisualizer(robot.model, robot.collision_model, robot.visual_model)
viz.initViewer(loadModel=True,open=True)

# show the frames we are interested in
frames = ["world", "tcp"]
frame_ids = []
for frame in frames:
    frame_ids.append(robot.model.getFrameId(frame))
viz.displayFrames(True, frame_ids)

# start at a specific configuration
robot.q0 = np.array([0.0, -1.9,  1.9, -1.6, -1.6, 0])
viz.updateFrames()

viz.display(robot.q0)


def visualize_balls(ball_positions):
    """
    Visualizes the balls in the meshcat viewer.
    """
    for i, pos in enumerate(ball_positions):
        viz.viewer[f"ball_{i}"].set_object(Sphere(BallSimulation.ball_radius), MeshPhongMaterial(color=COLORS[i % len(COLORS)]))
        viz.viewer[f"ball_{i}"].set_transform(translation_matrix(pos))

## Define the observer

In [ ]:
class TrajectoryPredictor:
    def __init__(self, dt):
        self.dt = dt
        self.states = []

    def step(self, ball_positions):
        pass

    def predict(self):
        pass

## Define the controller

In [ ]:
data = robot.model.createData()
pin.forwardKinematics(robot.model, data, robot.q0)
pin.updateFramePlacements(robot.model, data)
tcp_pose = data.oMf[robot.model.getFrameId("tcp")]
print("Tool center point pose in world frame:\n", tcp_pose)

class Controller:
    def __init__(self, estimator: TrajectoryPredictor):
        self.estimator = estimator

    def step(self, q, dq, ball_positions):
        """
        Computes the joint velocity commands to achieve a desired Cartesian velocity of the end-effector.

        Parameters:
            q (numpy.ndarray): Current joint configuration of the robot.
            dq (numpy.ndarray): Current joint velocity of the robot.
            ball_positions (list): Positions of the balls in the world frame.


        Returns:
            numpy.ndarray: Joint velocity commands.
        """

        self.estimator.step(ball_positions)

        return ACCELERATION_LIMIT

## The simulation loop

### Definition

In [ ]:
control_freq = 100  # Control frequency in Hz

def simulate(controller, simulation: BallSimulation):

    # Set parameters for the simulation
    render_freq = 32  # Rendering frequency in Hz
    sim_time = 3  # Total simulation time in seconds

    # Initialize joint positions and velocities
    q = robot.q0  # Initial joint configuration
    dq = 0.0  # Joint velocity vector

    # Start the simulation timer
    time_start = time.time()

    # Fix the random seed for reproducibility
    # np.random.seed(0)

    # Simulation loop
    dt = 1.0 / control_freq
    for t in np.arange(0, sim_time, dt):
        # read ball positions
        simulation.update(dt)
        ball_positions = simulation.get_positions()
        # Update the frames in the visualizer
        visualize_balls(ball_positions)
        viz.updateFrames()


        # execute the controller function to compute the joint accelerations (ddq)
        ddq = controller.step(q, dq, ball_positions)

        # Update the joint positions
        np.clip(ddq, -ACCELERATION_LIMIT, ACCELERATION_LIMIT)
        dq = dq + ddq * dt
        np.clip(dq, -VELOCITY_LIMIT, VELOCITY_LIMIT)
        q = pin.integrate(robot.model, q, dq * dt)
        np.clip(q, LOWER_POSITION_LIMIT, UPPER_POSITION_LIMIT)

        # Update the visualization at the specified rendering frequency
        if t % (1 / render_freq) < 1 / control_freq:

            # Display the robot's current configuration in the visualizer
            viz.display(q)

            # Calculate the remaining time to sleep (accounting for computation time)
            time_end = time.time()
            time_diff = time_end - time_start
            time_sleep = max(0, 1 / render_freq - time_diff)

            # Sleep for the remaining time or skip if computation took too long
            time.sleep(time_sleep)

            # Reset the simulation timer
            time_start = time.time()

### Execution

In [ ]:
ball_simulation = BallSimulation()
estimator = TrajectoryPredictor(control_freq)
controller = Controller(estimator)

simulate(controller, ball_simulation)

## Plots